# 01 — Memory limits: bandwidth, capacity, and latency

**Question.** How much useful memory traffic can the node sustain, how does the working set change that rate, and what does remote placement cost?

Flat mode explicitly selects DDR or HBM. Cache mode measures the effective memory system at a stated working set. GB/s is STREAM-model traffic, not a memory-controller counter. GiB describes allocation size. Every point below uses five steady samples; error bars show sample standard deviation. Each figure cell saves its displayed Plotly object as SVG and PNG.


In [ ]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench.analysis import (load_measurements, select_measurements, export_figure,
    memory_table, compute_table, locality_table, scaling_table, knees_table,
    theoretical_table, concurrent_table, bandwidth_scaling, cache_working_set,
    locality_figure, compute_scaling, fp64_rooflines, dual_scaling)
pio.renderers.default = 'notebook'
validation = json.loads((ROOT / 'data/processed/validation.json').read_text())
assert validation['status'] == 'passed', validation['errors']
assert validation['dataset_sha256'] == hashlib.sha256((ROOT / 'data/processed/measurements.csv').read_bytes()).hexdigest()
assert validation['selection_sha256'] == hashlib.sha256((ROOT / 'configs/analysis.json').read_bytes()).hexdigest()
df = select_measurements(load_measurements(ROOT / 'data/processed/measurements.csv'))
NOTEBOOK = '01_memory_limits'
def table(frame):
    shown = frame.copy()
    if 'implementation' in shown:
        shown['implementation'] = shown['implementation'].replace({
            'native_amx_4_tiles': 'Native AMX tiles',
            'onednn_numactl_only_omp_affinity_unset': 'oneDNN GEMM',
            'native_avx512': 'AVX-512'})
    if 'metric' in shown:
        shown['metric'] = shown['metric'].replace({'bandwidth_gbs':'GB/s','performance_gops':'GFLOP/s','latency_ns':'ns/load'})
    shown = shown.rename(columns={'memory_mode':'Mode','socket':'Socket','memory_target':'Memory',
        'locality':'Placement','working_set_gib':'Working set (GiB)','implementation':'Compute path',
        'dtype':'Precision','m':'Matrix size','n_samples':'Samples','efficiency':'Scaling efficiency'})
    display(shown.round(3).style.set_uuid(NOTEBOOK).hide(axis='index'))


## 1. Sustainable bandwidth and core count

**Conditions.** Triad (`A = B + 3C`), 8 GiB across all three arrays, local placement, one worker per physical core. The panels keep sockets separate. Increasing the number of cores reveals how much parallelism is needed to approach the measured sustainable rate.

In [ ]:
fig = bandwidth_scaling(df)
export_figure(fig, 'bandwidth_scaling', NOTEBOOK)

In [ ]:
m = memory_table(df)
full = m[m.memory_mode.eq('flat') | m.working_set_gib.round(0).isin([16,96])]
table(full[['memory_mode','socket','memory_target','working_set_gib','median','std','n_samples']].rename(columns={'median':'median_GB_s','std':'std_GB_s'}))
p = m[m.memory_mode.eq('flat')].pivot(index='socket', columns='memory_target', values='median')
for socket, row in p.iterrows():
    display(Markdown(f"**Result, socket {socket}.** HBM sustains {row['hbm']:.1f} GB/s and DDR {row['ddr']:.1f} GB/s: {row['hbm']/row['ddr']:.2f}× for this matched Triad workload."))


**Implication.** Use the measured rate for the relevant socket and memory placement when estimating the performance of a bandwidth-dominated workload. These rates include the kernel's traffic convention; write allocation and physical bus traffic can differ.

## 2. Cache capacity response

**Question.** How does bandwidth change as the active working set approaches and exceeds nominal HBM capacity?

**Conditions.** 56 cores on one socket at a time, local DDR-backed allocation, 8–128 GiB total active bytes. Both socket curves are shown. The 64 GiB marker is a capacity reference, not a fitted transition or measured hit-rate boundary.

In [ ]:
fig = cache_working_set(df)
export_figure(fig, 'cache_capacity', NOTEBOOK)

In [ ]:
a = m[m.memory_mode.eq('cache')]
for socket in [0,1]:
    p = a[a.socket.eq(socket)].set_index(a[a.socket.eq(socket)].working_set_gib.round().astype(int))
    small, large = p.loc[16,'median'], p.loc[96,'median']
    display(Markdown(f"**Result, socket {socket}.** Effective bandwidth changes from {small:.1f} GB/s at 16 GiB to {large:.1f} GB/s at 96 GiB ({large/small:.1%} of the smaller-working-set rate)."))


**Implication.** Cache-mode bandwidth must be specified together with working-set size. A small-allocation bandwidth result cannot describe an application whose active memory exceeds HBM capacity. Socket differences are observations; this experiment does not identify their cause.

## 3. NUMA placement cost

**Question.** Does HBM's local bandwidth advantage survive when memory belongs to the other socket?

**Conditions.** Flat mode, 56 cores, 8 GiB Triad. “Local” and “remote” refer to the memory socket relative to the CPU socket. The workload and CPU mask are held constant.

In [ ]:
fig = locality_figure(df, 'bandwidth_gbs')
export_figure(fig, 'numa_bandwidth', NOTEBOOK)

In [ ]:
loc = locality_table(df)
table(loc[loc.memory_mode.eq('flat') & loc.metric.eq('bandwidth_gbs')][['socket','memory_target','locality','median','std']].rename(columns={'median':'median_GB_s','std':'std_GB_s'}))

**Implication.** A NUMA memory policy is part of the achievable bandwidth configuration. Remote-memory rates must not be substituted with local HBM rates.

## 4. Idle dependent-load latency

**Question.** What is the cost of an individual dependent memory access?

**Conditions.** One core, a 1 GiB randomized cycle of 64-byte nodes, seed 7, a full-chain warm-up, five timed windows. Each next pointer depends on the previous load. This working set exceeds the last-level cache. Results include address translation and loop overhead and describe this access pattern under idle conditions, not loaded latency.

In [ ]:
fig = locality_figure(df, 'latency_ns')
export_figure(fig, 'memory_latency', NOTEBOOK)

In [ ]:
table(loc[loc.metric.eq('latency_ns')][['memory_mode','socket','memory_target','locality','working_set_gib','median','std']].rename(columns={'median':'median_ns_per_load','std':'std_ns_per_load'}))

**Implication.** Bandwidth and latency describe different limits: a dependent access stream cannot exploit the parallelism used by Triad. Cache-mode latency is an effective result at 1 GiB; it does not measure DDR-only latency or latency above HBM capacity.

## 5. Can DDR and HBM bandwidth be combined?

**Conditions.** Two simultaneously timed 28-core workers on disjoint CPU sets, each with an 8 GiB allocation. The isolated controls use those exact CPU sets. The combined rate counts total work over one shared interval. The 56-core HBM reference has one 8 GiB allocation, so it answers which configuration is faster rather than holding aggregate allocation size constant.

In [ ]:
table(concurrent_table(df))

**Interpretation.** Compare the combined row with the all-HBM row before proposing split-tier execution for raw throughput. Component rates show interference relative to the same-core controls. This experiment measures two independent streams; it does not establish the speed of a single application that mixes DDR and HBM accesses.

**Evidence.** Source paths and campaign identities are retained in `data/processed/measurements.csv`; the exact table exports are in `data/processed/thesis/`. Validation and excluded attempts are documented in the validation appendix. Continue with compute limits.